In [1]:
# !pip install diffusers transformers accelerate torch torchvision pillow matplotlib

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from diffusers import StableDiffusionPipeline
from PIL import Image

/home/bianca_cal/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

Using device: cuda


In [4]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to(device)

Loading pipeline components...: 100%|██████████| 7/7 [00:09<00:00,  1.30s/it]


In [5]:
prompt = "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution"

In [6]:
seed = 42

generator = torch.manual_seed(seed)

num_steps = 20

In [7]:
saved_latents = {}
target_steps = [4, 10, 16, 20]

In [8]:
def capture_latents_callback(pipe, step_index, timestep, callback_kwargs):
    
    latents = callback_kwargs["latents"]

    current_step = step_index + 1

    if current_step in target_steps:
        saved_latents[current_step] = latents.detach().clone()

        print(f"Latents saved at step {current_step}")

    return callback_kwargs

In [9]:
result = pipe(
    prompt=prompt,
    num_inference_steps=num_steps,
    generator=generator,
    
    callback_on_step_end=capture_latents_callback,
    callback_on_step_end_tensor_inputs=["latents"]
)

 20%|██        | 4/20 [00:01<00:04,  3.71it/s]

Latents saved at step 4


 50%|█████     | 10/20 [00:02<00:01,  5.94it/s]

Latents saved at step 10


 80%|████████  | 16/20 [00:03<00:00,  6.44it/s]

Latents saved at step 16


100%|██████████| 20/20 [00:04<00:00,  5.00it/s]

Latents saved at step 20


In [10]:
print(saved_latents.keys())

dict_keys([4, 10, 16, 20])


In [11]:
for step, latent in saved_latents.items():
    print(f"Step {step}: shape = {latent.shape}")

Step 4: shape = torch.Size([1, 4, 64, 64])
Step 10: shape = torch.Size([1, 4, 64, 64])
Step 16: shape = torch.Size([1, 4, 64, 64])
Step 20: shape = torch.Size([1, 4, 64, 64])


Los tensores capturados presentan una dimensión de [1,4,64,64], correspondiente al espacio latente utilizado por Stable Diffusion v1.5.
La U-Net no opera directamente sobre imágenes RGB, sino sobre representaciones comprimidas generadas por el VAE Encoder.
Los 4 canales latentes contienen información semántica y estructural de la imagen, mientras que la resolución espacial reducida (64×64) permite disminuir significativamente el costo computacional durante el proceso iterativo de denoising.

# Referencia:
* https://huggingface.co/docs/diffusers/index